In [152]:
# Initialize Databricks Connect when running locally (spark/dbutils are pre-injected on Databricks)
try:
    spark
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().profile("fe-vm-mkgs-databricks-demos").getOrCreate()

In [153]:
# Widget definitions (on Databricks) / config (Databricks Connect - widgets API limited)
_scope = "epic_on_fhir_oauth_keys"
_client_key = "client_id"
_token_url = "https://fhir.epic.com/interconnect-fhir-oauth/oauth2/token"
_algo = "RS384"

try:
    dbutils.widgets.text("epic_secrets_scope", _scope, "Epic Secrets Scope")
    dbutils.widgets.text("client_id_dbs_key", _client_key, "Epic Client ID DB Secrets Key")
    dbutils.widgets.text("token_url", _token_url, "Epic Token URL")
    dbutils.widgets.text("algo", _algo, "Epic Token Enrcyption Algorithm")
except Exception:
    pass  # Databricks Connect: widget creation may not be fully supported

Box(children=(Label(value='Epic Secrets Scope'), Text(value='epic_on_fhir_oauth_keys')))

Box(children=(Label(value='Epic Client ID DB Secrets Key'), Text(value='client_id')))

Box(children=(Label(value='Epic Token URL'), Text(value='https://fhir.epic.com/interconnect-fhir-oauth/oauth2/…

Box(children=(Label(value='Epic Token Enrcyption Algorithm'), Text(value='RS384')))

In [154]:
# Databricks Connect doesn't support widgets.getAll(); use config fallback for local dev
try:
    widget_values = dbutils.widgets.getAll()
except AttributeError:
    widget_values = {
        "epic_secrets_scope": _scope,
        "client_id_dbs_key": _client_key,
        "token_url": _token_url,
        "algo": _algo,
    }

for name, value in widget_values.items():
    print(f"{name} = {value}")

epic_secrets_scope = epic_on_fhir_oauth_keys
client_id_dbs_key = client_id
token_url = https://fhir.epic.com/interconnect-fhir-oauth/oauth2/token
algo = RS384


In [155]:
import time
import jwt
import requests

In [156]:
CLIENT_ID = dbutils.secrets.get(scope = widget_values["epic_secrets_scope"], key = widget_values["client_id_dbs_key"])
TOKEN_URL = widget_values["token_url"]
PRIVATE_KEY = dbutils.secrets.get(scope = widget_values["epic_secrets_scope"], key = "private_key")
ALGO = widget_values["algo"]
EPIC_KID = dbutils.secrets.get(scope=widget_values["epic_secrets_scope"], key="kid")

In [157]:
# print(f"""
# CLIENT_ID = {CLIENT_ID}
# TOKEN_URL = {TOKEN_URL}
# PRIVATE_KEY = {PRIVATE_KEY}
# ALGO = {ALGO}
# EPIC_KID = {EPIC_KID}
# """)

In [158]:
def get_access_token():

	now = int(time.time())
	payload = {
		"iss": CLIENT_ID
		, "sub": CLIENT_ID
		, "aud": TOKEN_URL
		, "jti": str(now)
		, "exp": now + 300  # Token valid for 5 minutes
	}
	
	jwt_headers = {
        "kid": EPIC_KID,
        "typ": "JWT"
	}

	jwt_token = jwt.encode(payload, PRIVATE_KEY, algorithm=ALGO, headers=jwt_headers)

	headers = { "Content-Type": "application/x-www-form-urlencoded" }
	data = {
		"grant_type": "client_credentials"
		, "client_assertion_type": "urn:ietf:params:oauth:client-assertion-type:jwt-bearer"
		, "client_assertion": jwt_token
		# , "scope": "system/Patient.read"
	}

	response = requests.post(TOKEN_URL, data=data, headers=headers)
	response.raise_for_status()
	# return response.json()["access_token"]
	return response.json()

In [159]:
import json

token = get_access_token()
# print(json.dumps(token, indent=2))

In [160]:
token = get_access_token()
headers = {
	"Authorization": f"Bearer {token['access_token']}"
    ,"Accept": "application/json"
}
url = "https://fhir.epic.com/interconnect-fhir-oauth/api/FHIR/R4/Patient/T1wI5bk8n1YVgvWk9D05BmRV0Pi3ECImNSK8DKyKltsMB"
response = requests.get(url, headers=headers)
response.raise_for_status()
print(response.status_code, response.text)

200 {"resourceType":"Patient","id":"eOPSKZbz6YIgoilQprzPy0Q3","extension":[{"valueCodeableConcept":{"coding":[{"system":"urn:oid:1.2.840.114350.1.13.0.1.7.10.698084.130.657370.19999000","code":"female","display":"female"}],"text":"Female"},"url":"http://open.epic.com/FHIR/StructureDefinition/extension/legal-sex"},{"valueCodeableConcept":{"coding":[{"system":"urn:oid:1.2.840.114350.1.13.0.1.7.10.698084.130.657370.19999000","code":"female","display":"female"}]},"url":"http://open.epic.com/FHIR/StructureDefinition/extension/sex-for-clinical-use"},{"extension":[{"valueCoding":{"system":"urn:oid:2.16.840.1.113883.6.238","code":"2131-1","display":"Other Race"},"url":"ombCategory"},{"valueString":"Other","url":"text"}],"url":"http://hl7.org/fhir/us/core/StructureDefinition/us-core-race"},{"extension":[{"valueString":"Unknown","url":"text"}],"url":"http://hl7.org/fhir/us/core/StructureDefinition/us-core-ethnicity"},{"valueCode":"248152002","url":"http://hl7.org/fhir/us/core/StructureDefinition

In [161]:
response.json()

{'resourceType': 'Patient',
 'id': 'eOPSKZbz6YIgoilQprzPy0Q3',
 'extension': [{'valueCodeableConcept': {'coding': [{'system': 'urn:oid:1.2.840.114350.1.13.0.1.7.10.698084.130.657370.19999000',
      'code': 'female',
      'display': 'female'}],
    'text': 'Female'},
   'url': 'http://open.epic.com/FHIR/StructureDefinition/extension/legal-sex'},
  {'valueCodeableConcept': {'coding': [{'system': 'urn:oid:1.2.840.114350.1.13.0.1.7.10.698084.130.657370.19999000',
      'code': 'female',
      'display': 'female'}]},
   'url': 'http://open.epic.com/FHIR/StructureDefinition/extension/sex-for-clinical-use'},
  {'extension': [{'valueCoding': {'system': 'urn:oid:2.16.840.1.113883.6.238',
      'code': '2131-1',
      'display': 'Other Race'},
     'url': 'ombCategory'},
    {'valueString': 'Other', 'url': 'text'}],
   'url': 'http://hl7.org/fhir/us/core/StructureDefinition/us-core-race'},
  {'extension': [{'valueString': 'Unknown', 'url': 'text'}],
   'url': 'http://hl7.org/fhir/us/core/Struc

In [162]:
"""
Class to handle authentication
"""

import datetime, jwt, requests, json, zoneinfo
from uuid import uuid4

class EpicApiAuth(requests.auth.AuthBase):
  def __init__(self, 
               client_id, 
               private_key, 
               kid,
               algo,
               auth_location = "https://fhir.epic.com/interconnect-fhir-oauth/oauth2/token"):
    self.__client_id = client_id
    self.__private_key = private_key
    self.__kid = kid
    self.__algo = algo
    self.auth_location = auth_location
    self.__token = None
    self.token_expiry = None

  def get_token(self,
                now = None,
                expiration = None,
                timeout=30):
    now = (now if now is not None else datetime.datetime.now(zoneinfo.ZoneInfo("America/New_York")))
    expiration = (expiration if expiration is not None else datetime.datetime.now(zoneinfo.ZoneInfo("America/New_York")) + datetime.timedelta(minutes=5))
    if self.__token is None or now >= self.token_expiry:
      t = self.generate_token(now, expiration, timeout)
      t.raise_for_status()
      self.__token = json.loads(t.text)
      self.token_expiry = expiration
    return self.__token
  
  def __call__(self, r):
    r.headers['Authorization'] = 'Bearer %s' % self.get_token()['access_token']
    r.headers['Accept'] = 'application/json'
    r.headers['Content-Type'] = 'application/json'
    return r

  """
    Provide authentication to EPIC on FHIR OAuth2 and return valid token
      @param expiration = the datetime when the token expires, default 5 minutes
      @param timeout = seconds to timeout request, default 30 
  """
  def generate_token(self,
                     now = None,
                     expiration = None,
                     timeout=30):
    now = (now if now is not None else datetime.datetime.now(zoneinfo.ZoneInfo("America/New_York")))
    expiration = (expiration if expiration is not None else datetime.datetime.now(zoneinfo.ZoneInfo("America/New_York")) + datetime.timedelta(minutes=5))
    return requests.post(self.auth_location, 
        data= {
        'grant_type': 'client_credentials',
        'client_assertion_type': 'urn:ietf:params:oauth:client-assertion-type:jwt-bearer',
        'client_assertion': jwt.encode(
           {
              'iss': self.__client_id,
              'sub': self.__client_id,
              'aud': self.auth_location,
              'exp': int(expiration.timestamp()),
              'iat': int(now.timestamp()),
              'jti': uuid4().hex,
          },
          self.__private_key,
          algorithm=self.__algo,
          headers={
            'kid': self.__kid,
            'alg': self.__algo,
            'typ': 'JWT',
          })
      }, timeout=timeout)
    
  def can_connect(self):
    return (self.generate_token().status_code == 200)

In [163]:
epicAuth = EpicApiAuth(
  client_id = CLIENT_ID,
  private_key = PRIVATE_KEY,
  kid = EPIC_KID,
  algo = ALGO,
  auth_location = TOKEN_URL)

In [164]:
epicAuth.can_connect()

True

In [165]:
class EpicApiRequest:
    def __init__(self, auth, base_url):
        self.auth = auth
        self.base_url = base_url

    def make_request(self, http_method, resource, action, data=None):
        response = getattr(requests, http_method)(f"{self.base_url}{resource}/{action}", data=data, auth=self.auth)
        
        return {'request':
                {
                    'http_method': http_method,
                    'url': f"{self.base_url}{resource}/{action}",
                    'data': ('' if data is None else data)
                },
                'response': {
                    'response_status_code': response.status_code, 
                    'response_time_seconds': (response.elapsed.microseconds / 1000000),
                    'response_headers': response.headers,
                    'response_text': response.text,
                    'response_url': response.url
                }
            }

In [166]:
epic_api = EpicApiRequest(
    auth = epicAuth,
    base_url = "https://fhir.epic.com/interconnect-fhir-oauth/api/FHIR/R4/"
)

In [167]:
epic_api.make_request(
    http_method = "get",
    resource = "Patient",
    action = "T1wI5bk8n1YVgvWk9D05BmRV0Pi3ECImNSK8DKyKltsMB"
)

{'request': {'http_method': 'get',
  'url': 'https://fhir.epic.com/interconnect-fhir-oauth/api/FHIR/R4/Patient/T1wI5bk8n1YVgvWk9D05BmRV0Pi3ECImNSK8DKyKltsMB',
  'data': ''},
 'response': {'response_status_code': 200,
  'response_time_seconds': 0.416876,
  'response_headers': {'Cache-Control': 'no-cache,no-store', 'Pragma': 'no-cache', 'Content-Length': '1583', 'Content-Type': 'application/json; charset=utf-8', 'Content-Encoding': 'gzip', 'Expires': '-1', 'ServerMetrics': '{"BlockReads":1,"BlockWrites":2,"BlocksAllocated":1,"DBTime":33,"DatabaseCPUTime":34,"ECFNetworkTime":2,"ECFRequestCount":1,"ECFRequestTime":35,"GREF":3249,"JournalEntries":14,"LockWait":0.03,"LocksFailed":0,"LocksGranted":0,"MCommands":184449,"MemoryDifference":0,"NetworkCacheMisses":0,"NetworkUpdates":0,"WorkflowEventBlockReads":1,"WorkflowEventDBTime":33,"WorkflowEventECFNetworkTime":1,"WorkflowEventECFRequestCount":1,"WorkflowEventGREF":3249}', 'E-Tag': '505020769', 'Strict-Transport-Security': 'max-age=31536000; 

In [168]:
# Epic requires encounter for Observation create. Search for encounters for our patient.
# Prefer recent encounters (date >= today) so effectiveDateTime "now" falls within encounter period.
# Use 'dt' alias to avoid shadowing datetime module used by EpicApiAuth
from datetime import datetime as dt, timezone

patient_id = "T1wI5bk8n1YVgvWk9D05BmRV0Pi3ECImNSK8DKyKltsMB"
today = dt.now(timezone.utc).strftime("%Y-%m-%d")
enc_resp = epic_api.make_request(
    http_method="get",
    resource="Encounter",
    action=f"?patient=Patient/{patient_id}&date=ge{today}"
)
enc_bundle = json.loads(enc_resp["response"]["response_text"])
encounters = enc_bundle.get("entry", [])

# Fallback: if no recent encounters, use any encounter (effectiveDateTime will use a time within that encounter)
if not encounters:
    enc_resp = epic_api.make_request(http_method="get", resource="Encounter", action=f"?patient=Patient/{patient_id}")
    enc_bundle = json.loads(enc_resp["response"]["response_text"])
    encounters = enc_bundle.get("entry", [])

if not encounters:
    raise ValueError("No encounters found for patient - Observation create requires a valid encounter.")

# Use first encounter; if from the past, we'll use a unique time within its period
encounter = encounters[0]["resource"]
encounter_id = encounter["id"]
enc_period = encounter.get("period", {})
enc_start = enc_period.get("start")
enc_end = enc_period.get("end")

In [169]:
# FHIR R4 Observation payload - use unique effectiveDateTime to avoid Epic 59189 "Reading already exists"
# Use 'dt' alias to avoid shadowing datetime module used by EpicApiAuth
from datetime import datetime as dt, timezone, timedelta

now_utc = dt.now(timezone.utc)
# If encounter has a period, use enc_start + offset (unique per run) so we stay within period
if enc_start:
    base = dt.fromisoformat(enc_start.replace("Z", "+00:00")) if "T" in enc_start else dt.fromisoformat(enc_start + "T12:00:00+00:00")
    offset_sec = int(now_utc.timestamp()) % 3600  # unique per hour
    effective_dt = (base + timedelta(seconds=offset_sec)).strftime("%Y-%m-%dT%H:%M:%SZ")
else:
    effective_dt = now_utc.strftime("%Y-%m-%dT%H:%M:%SZ")

observation_payload = {
    "resourceType": "Observation",
    "status": "final",
    "category": [
        {
            "coding": [
                {
                    "system": "http://hl7.org/fhir/observation-category",
                    "code": "vital-signs",
                    "display": "Vital Signs"
                }
            ],
            "text": "Vital Signs"
        }
    ],
    "code": {
        "coding": [
            {
                "system": "urn:oid:1.2.840.114350.1.13.0.1.7.2.707679",
                "code": "8",
                "display": "Heart Rate"
            }
        ],
        "text": "Heart Rate"
    },
    "subject": {
        "reference": f"Patient/{patient_id}"
    },
    "encounter": {
        "reference": f"Encounter/{encounter_id}"
    },
    "effectiveDateTime": effective_dt,
    "valueQuantity": {
        "value": 75
    }
}

In [170]:
obs_resp = epic_api.make_request(
    http_method="post",
    resource="Observation",
    action="",
    data=json.dumps(observation_payload)
)

# Print OperationOutcome when 4xx/5xx
sc = obs_resp["response"]["response_status_code"]
if sc >= 400:
    try:
        oo = json.loads(obs_resp["response"]["response_text"])
        if oo.get("resourceType") == "OperationOutcome":
            for issue in oo.get("issue", []):
                diag = issue.get("diagnostics", "")
                details = issue.get("details", {}).get("text", "")
                loc = issue.get("location", [])
                print(f"Epic error: {details} | diagnostics: {diag} | location: {loc}")
    except Exception:
        pass
obs_resp

{'request': {'http_method': 'post',
  'url': 'https://fhir.epic.com/interconnect-fhir-oauth/api/FHIR/R4/Observation/',
  'data': '{"resourceType": "Observation", "status": "final", "category": [{"coding": [{"system": "http://hl7.org/fhir/observation-category", "code": "vital-signs", "display": "Vital Signs"}], "text": "Vital Signs"}], "code": {"coding": [{"system": "urn:oid:1.2.840.114350.1.13.0.1.7.2.707679", "code": "8", "display": "Heart Rate"}], "text": "Heart Rate"}, "subject": {"reference": "Patient/T1wI5bk8n1YVgvWk9D05BmRV0Pi3ECImNSK8DKyKltsMB"}, "encounter": {"reference": "Encounter/e0u1fd.jUCNqz8ZQuTaMtsQ3"}, "effectiveDateTime": "2016-01-05T06:04:58Z", "valueQuantity": {"value": 75}}'},
 'response': {'response_status_code': 201,
  'response_time_seconds': 0.566028,
  'response_headers': {'Cache-Control': 'no-cache,no-store', 'Pragma': 'no-cache', 'Content-Type': 'application/json; charset=utf-8', 'Expires': '-1', 'Location': 'Observation/fXNbd6f-aj5V5pZC2e8uXzImGQZ7eWDT4c01Mr

In [186]:
# Retrieve the created Observation via GET
if obs_resp["response"]["response_status_code"] == 201:
    location = obs_resp["response"]["response_headers"].get("Location", "")
    obs_id = location.split("/")[-1]
else:
    print("Observation was not created (status != 201); nothing to retrieve")

In [189]:
try:
    new_obs = epic_api.make_request(
        http_method="get",
        resource="Observation",
        action=obs_id
    )
    if new_obs["response"]["response_status_code"] == 200:
        obs_resource = json.loads(new_obs["response"]["response_text"])
        print(json.dumps(obs_resource, indent=2))
    else:
        print(json.dumps(new_obs, indent=2))
except Exception as e:
    print(f"Failed to retrieve Observation: {e}")

{
  "resourceType": "Observation",
  "id": "fXNbd6f-aj5V5pZC2e8uXzImGQZ7eWDT4c01MrWwrGS84",
  "extension": [
    {
      "valueIdentifier": {
        "system": "urn:oid:1.2.840.114350.1.13.0.1.7.2.707684",
        "value": "31020"
      },
      "url": "http://open.epic.com/FHIR/StructureDefinition/extension/template-id"
    }
  ],
  "status": "final",
  "category": [
    {
      "coding": [
        {
          "system": "http://terminology.hl7.org/CodeSystem/observation-category",
          "code": "vital-signs",
          "display": "Vital Signs"
        }
      ],
      "text": "Vital Signs"
    }
  ],
  "code": {
    "coding": [
      {
        "system": "urn:oid:1.2.840.114350.1.13.0.1.7.2.707679",
        "code": "8",
        "display": "Pulse"
      },
      {
        "system": "http://open.epic.com/FHIR/StructureDefinition/observation-flowsheet-id",
        "code": "t0nle2B9NVDaJJUDo0FtQrA0",
        "display": "Pulse"
      },
      {
        "system": "urn:oid:1.2.246.537.6.9